# 논문 C 「한국 HSK 10단위 연도별 실행세율 자료의 구축」이 쓴 프로그램

이 노트북은 논문 C(`실행세율_자료구축.md`)의 본문과 부록 A(재현 절차)가 쓴 파이썬 프로그램 전부를 한곳에 모은 것이다. 셀은 실행 순서(부록 표 A3)대로 두었고, 각 셀은 저장소의 스크립트 파일을 **그대로** 옮긴 것이다(첫 줄 주석에 파일 경로). 저장소에 공개하는 것은 `scripts/01`·`02`이고 `research/scripts/17·21·24·25·26`은 로컬에만 두므로 이 노트북이 그 공개 기록을 겸한다.

**실행 환경.** conda 환경 `kcsdb`(duckdb·pandas·requests·olefile·pymupdf·openpyxl). 무역통계 DB와 HSK 별표는 KCSDB2 저장소에서 읽기 전용으로 쓴다(환경변수 `KCSDB2_ROOT`·`KCSDB2_PATH`). 이 노트북을 위에서 아래로 그냥 실행하면 포털·법령센터·Comtrade·ECOS에 실제로 요청을 보내므로(합쳐 몇 시간, 원본 캐시가 있으면 적재만), 다시 하려는 셀만 골라 돌린다. 각 셀의 `if __name__ == "__main__":` 아래가 진입점이다.

**저장된 프로그램이 없는 작업 셋.** 사전세액심사 목록은 브라우저 페이지 컨텍스트에서 자바스크립트로 받았고(그 코드는 아래에 그대로 둔다) JSON을 CSV로 옮기는 것은 대화형이었다. 품목분류 적용기준 규칙 별표는 PDF 텍스트를 뽑은 뒤 손으로 표에 옮겼다. 유통이력관리 고시 별표 48판은 아래의 HWP·HWPX 읽기 함수로 텍스트를 뽑은 뒤 대화형으로 판본별·구간·통합 표를 만들었으며, 그 규칙은 부록 A2.4에 있다. 양허관세 별표 1과 ECOS 환율도 처음에는 대화형이었으나 2026-09-13에 스크립트(25·26)로 복원해 옛 산출과 전부 같음을 확인했다.

| 순서 | 무엇 | 셀 | 산출 |
|---|---|---|---|
| 1 | 관세율표·주요세율보기 스무 해 | `scripts/01_fetch_tariff.py` | `kcstariff.duckdb`의 `tariff_code`·`tariff_rate`·`dim_rate_cd`·`meta_fetch` |
| 2 | 실행세율 | `scripts/02_build_applied_rate.py` | `fct_applied_rate`·`dim_origin_regime`, 법령 대조 표본 |
| 3 | 양허관세 별표 1 | `research/scripts/25_parse_concession_annex.py` | `양허관세_별표1_2025.csv` |
| 4 | 품목분류 규칙 별표 | PDF 텍스트 추출 스니펫 | 텍스트(표는 손으로) |
| 5 | 유통이력 별표 48판 | HWP·HWPX 읽기 함수 | 텍스트(표는 대화형) |
| 6 | 사전세액심사 월별 목록 | 자바스크립트(페이지 컨텍스트) | JSON(CSV는 대화형) |
| 7 | 품목 상세(연구 코드만) | `research/scripts/17_fetch_item_detail.py` | `품목상세_세율_2012_2026.csv`·`품목상세_부가정보_2012_2026.csv` |
| 8 | 미러 통계 | `research/scripts/21_fetch_comtrade_mirror.py` | `comtrade_mirror_hs6_2012_2024.csv` |
| 9 | 환율 | `research/scripts/26_fetch_ecos_fx.py` | `환율_월별_USDKRW.csv` |
| 10 | 법령 대조 표본 214개 | `research/scripts/24_check_tariff_annex.py` | `실행세율_법령대조.csv`의 확인 열, `_요약.csv` |

본문·부록의 인용 수치는 같은 폴더의 실행용 노트북 `논문C_검증.ipynb`(빌더 `cache/build_nb_paperC.py`, 셀은 세 논문 공용 셀 창고 `cache/cells.py`의 §1·§1b·§6)가 등록하고 대조한다(이 노트북과는 별개).

## 1. 관세율표·주요세율보기 — 세율 두 화면 스무 해

부록 A1.1·A1.2. 캐시가 있으면 `--parse-only`, 검증만 하려면 `--verify-only`.

In [ ]:
# ===== 파일: scripts/01_fetch_tariff.py (그대로 옮김, 23,775바이트) =====
"""
01_fetch_tariff.py — 연도별 HSK 10단위 실행세율 수집 (관세법령정보포털)

무엇을 푸는가:
  KCSDB2에는 관세율이 없다. 품목분류와 세율 차이를 함께 보려면(세율이 높은 품목의 수입이
  세율 낮은 비슷한 품목으로 옮겨 가는가) 코드마다 그해 적용된 세율이 있어야 한다.
  공공데이터포털의 품목번호별 관세율표(15051179)는 현행판만 있어 과거를 못 준다.

  관세법령정보포털은 2002년부터 연도별로 두 화면을 준다. 둘을 합쳐야 실행세율이 된다.
    관세율표(openULS0201005Q)   기본세율 + 탄력·양허 세율(WTO 협정세율 C, 농림축산물
                                양허관세 W1·W2, 조정관세 L, 할당관세 P, 특별긴급관세 T,
                                아시아·태평양 협정세율 E 등). 연중 변경 표시는 없다.
    주요세율보기(openULS0201017Q) 기본·WTO·아시아태평양 + FTA 일곱 상대(중국·EU·미국·
                                아세안·인도·베트남·캐나다). 연중에 바뀌면 기간을 나눠
                                보여 준다(한-EU FTA는 7월 1일 인하). 조정·할당·양허는 없다.
  그 밖의 FTA(칠레·EFTA·호주 등)는 품목 상세 화면에만 있어 코드마다 따로 받아야 하므로
  여기서는 받지 않는다. 연구 대상 코드가 정해지면 그것만 받는다.

받는 방법:
  두 화면 모두 hsfdCd에 류(2자리)를 주면 그 류 전체가 온다. 연도당 97회씩이다.
  관세율표 화면은 KCSDB2의 `03j_fetch_hsk_table.py`와 같은 요청이라 캐시(data/raw/clip_hsk)를
  함께 쓴다 — 2007~2010년은 이미 받아 두었다.

입력: 없음 (포털에서 받는다)
출력:
  data/raw/clip_hsk/<연도>/<류>.html            관세율표 캐시 (03j와 공유)
  data/raw/clip_tariff_main/<연도>/<류>.html    주요세율보기 캐시
  data/processed/kcstariff.duckdb               별도 DB 파일 (KCSDB2와 ATTACH로 결합)
    tariff_code   year, hs10, name_ko, source              그해 관세율표의 10단위 코드
    tariff_rate   year, hs10, rate_cd, rate_txt, adval, specific, valid_from, valid_to, source
                  source는 table(관세율표) · main(주요세율보기의 FTA 열) · fill(관세율표 화면을
                  받지 못한 코드를 주요세율보기의 기본·WTO·아시아태평양 세율로 채운 것)
    dim_rate_cd   rate_cd, rate_nm, source
    meta_fetch    page, year, ryu, fetched_at, bytes

세율 값:
  rate_txt는 화면 문구 그대로다. adval은 종가세율(%), specific은 종량세액(원)이다.
  「270% 또는 6,210원」은 둘 다 채운다. 어느 쪽을 적용하는지는 여기서 정하지 않는다.
  FTA 세율은 특혜를 신청했을 때 적용 가능한 세율이지 실제 납부 세율이 아니다.
  포털은 10단위 세율이 참고용이고 법적 효력이 없다고 밝힌다.

검증:
  ① 두 화면의 코드 집합이 같은가 ② 두 화면에 함께 나오는 기본세율·WTO 세율이 같은가
  ③ 알려진 값 몇 개(건고추 270%, 고추다진양념 조정관세 45% 등) ④ 그해 수입액 중
  세율표에 코드가 있는 몫.

실행:
  python scripts\\01_fetch_tariff.py                       # 2007~2026 전부
  python scripts\\01_fetch_tariff.py --years 2025 --ryu 9 21
  python scripts\\01_fetch_tariff.py --parse-only          # 캐시에서 다시 적재만
"""

from __future__ import annotations

import argparse
import html
import logging
import os
import re
import sys
import time
from datetime import date, datetime
from pathlib import Path

import duckdb
import pandas as pd
import requests

PROJECT_ROOT = Path(__file__).resolve().parent.parent
# 원본 캐시는 크다(관세율표 700MB·주요세율 450MB). 이미 받은 캐시가 다른 곳에 있으면 KCSTARIFF_RAW로 가리킨다.
RAW = Path(os.environ.get("KCSTARIFF_RAW", PROJECT_ROOT / "data" / "raw"))
DB_OUT = PROJECT_ROOT / "data" / "processed" / "kcstariff.duckdb"
# 수입액 커버리지 검증은 KCSDB2(무역통계 DB)가 있을 때만 한다. 없으면 건너뛴다.
DB_TRADE = Path(os.environ.get("KCSDB2_PATH", PROJECT_ROOT.parent / "KCSDB2" / "data" / "processed" / "kcsdb.duckdb"))
LOG_DIR = PROJECT_ROOT / "logs"
OUT_DIR = PROJECT_ROOT / "outputs"          # 검증 수치 CSV(수집_검증.csv)
LOG_DIR.mkdir(exist_ok=True)

LOG_PATH = LOG_DIR / f"fetch_tariff_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(LOG_PATH, encoding="utf-8"), logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

BASE = "https://unipass.customs.go.kr/clip/hsinfosrch/"
PAGES = {
    "table": ("openULS0201005Q.do", RAW / "clip_hsk"),
    "main": ("openULS0201017Q.do", RAW / "clip_tariff_main"),
}
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Referer": "https://unipass.customs.go.kr/clip/index.do",
    "Content-Type": "application/x-www-form-urlencoded",
}
YEARS = range(2007, 2027)
RYU = range(1, 100)
ERROR_MARK = "프로그램 오류발생"

# 주요세율보기의 열 이름. 화면 머리글은 '중국'처럼 짧아 이름을 여기서 준다.
MAIN_NAMES = {
    "FCN1": "한ㆍ중국 FTA협정세율(선택1)", "FEU1": "한ㆍEU FTA협정세율(선택1)",
    "FUS1": "한ㆍ미 FTA 협정세율(선택1)", "FAS1": "한ㆍ아세안 FTA협정세율(선택1)",
    "FIN1": "한ㆍ인도 FTA협정세율(선택1)", "FVN1": "한ㆍ베트남 FTA협정세율(선택1)",
    "FCA1": "한ㆍ캐나다 FTA협정세율(선택1)",
}


def fetch(sess: requests.Session, page: str, year: int, ryu: int) -> str:
    """류 하나를 받는다. 관세율표는 03j와 캐시를 함께 쓰므로 요청 변수도 03j와 같게 둔다."""
    ymd = "20070101" if page == "table" else f"{year}0101"
    data = dict(cntyCd="KR", aplyYy=str(year), cntyNm="한국", compareCrrspndNation="KR",
                sctYear=ymd, hstdYear=ymd, manlOrgnTpcd="01", tabTpcd="3",
                sctCd="01", hstdCd=f"{ryu:02d}", hsfdCd=f"{ryu:02d}", searchVal=f"{ryu:02d}")
    r = sess.post(BASE + PAGES[page][0], data=data, timeout=60)
    r.raise_for_status()
    # 서버가 표를 절반쯤 보내다 오류 안내문으로 끝내는 일이 있다(2011년 제90류). 상태 코드는
    # 200이라 문구로 가려낸다 — 그대로 캐시하면 코드 수백 개가 조용히 빠진다.
    if ERROR_MARK in r.text:
        raise requests.RequestException("응답이 오류 안내문으로 끝났다")
    return r.text


def cell_text(x: str) -> str:
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", x))).replace(chr(0xa0), " ").strip()


def parse_rate(txt: str) -> tuple[float | None, float | None]:
    """'270% 또는 6,210원' → (270.0, 6210.0). 못 읽으면 None."""
    a = re.search(r"([\d.]+)\s*%", txt)
    s = re.search(r"([\d,]+(?:\.\d+)?)\s*원", txt)
    return (float(a.group(1)) if a else None,
            float(s.group(1).replace(",", "")) if s else None)


def parse_table(page: str, year: int) -> tuple[list, list]:
    """관세율표 화면 → (코드 행, 세율 행).

    10단위 행마다 기본세율 칸이 있고, 탄력·양허 세율은 표에 비어 있는 span을 페이지의
    스크립트(f_KorAdTax)가 행 번호(korAdTax_N)로 채운다. 행 번호로 둘을 잇는다.
    """
    codes, rates, rowno = [], [], {}
    for tr in re.findall(r"<tr[^>]*>(.*?)</tr>", page, re.S):
        h = re.search(r'name="hsSgn_Mn" value="(\d{10})"', tr)
        k = re.search(r'korAdTax_(\d+)', tr)
        if not (h and k):
            continue
        tds = [cell_text(x) for x in re.findall(r"<td[^>]*>(.*?)</td>", tr, re.S)]
        hs10 = h.group(1)
        rowno[k.group(1)] = hs10
        codes.append((year, hs10, tds[3] if len(tds) > 3 else ""))
        if len(tds) > 5 and tds[5]:
            rates.append((year, hs10, "A", tds[5], None, None, "table"))

    js = page[page.find("function f_KorAdTax"):]
    js = js[:js.find("</script>")]
    for blk in re.split(r'showID = "#korAdTax_" \+ "', js)[1:]:
        n = blk[:blk.find('"')]
        cds = re.findall(r'<span title="([^"]+)"><a href="#"><span class="textColor">(\w+)</span>', blk)
        vals = re.findall(r"htm2 \+= '<span><a href=\"#\">([^<]*)</a>", blk)
        if len(cds) != len(vals):
            logger.warning("  %d년 행 %s: 세율 구분 %d개와 값 %d개가 어긋난다", year, n, len(cds), len(vals))
        hs10 = rowno.get(n)
        if hs10 is None:
            continue
        for (nm, cd), v in zip(cds, vals):
            rates.append((year, hs10, cd, v.strip(), nm, None, "table"))
    return codes, rates


PERIOD = re.compile(r"(.*?)\s*-\s*\((\d{4}-\d{2}-\d{2})\s*~\s*(\d{4}-\d{2}-\d{2})?\)")


def parse_main(page: str, year: int) -> tuple[list, list]:
    """주요세율보기 화면 → 세율 행. 머리글 마지막 줄의 구분기호(A, C, FCN1 …)가 열 순서다.

    첫 줄에도 'E1'이라는 표시가 있어 머리글 전체에서 구분기호를 모으면 열이 하나 늘어난다.
    """
    thead = page[page.find("<thead"):page.find("</thead>")]
    last = re.findall(r"<tr[^>]*>(.*?)</tr>", thead, re.S)[-1]
    cols = [c for c in (cell_text(x) for x in re.findall(r"<th[^>]*>(.*?)</th>", last, re.S))
            if re.fullmatch(r"[A-Z][A-Z0-9]*", c)]
    out, names = [], []
    body = page[page.find("<tbody"):]
    for tr in re.findall(r"<tr[^>]*>(.*?)</tr>", body, re.S):
        tds = [cell_text(x) for x in re.findall(r"<td[^>]*>(.*?)</td>", tr, re.S)]
        if len(tds) < 4 + len(cols) or not (re.fullmatch(r"\d{4}", tds[0])
                                             and re.fullmatch(r"\d{2}", tds[1])
                                             and re.fullmatch(r"\d{4}", tds[2])):
            continue
        hs10 = tds[0] + tds[1] + tds[2]
        names.append((year, hs10, tds[3]))
        for cd, v in zip(cols, tds[4:4 + len(cols)]):
            if not v:
                continue
            segs = PERIOD.findall(v)
            if segs:
                for val, f, t in segs:
                    out.append((year, hs10, cd, val.strip(), None, (f, t or None), "main"))
            else:
                out.append((year, hs10, cd, v, None, None, "main"))
    return out, names


def collect(years, ryus, pages, delay, parse_only):
    sess = requests.Session()
    sess.headers.update(HEADERS)
    meta = []
    n_req = 0
    for y in years:
        for page in pages:
            cache = PAGES[page][1] / str(y)
            cache.mkdir(parents=True, exist_ok=True)
            for ryu in ryus:
                f = cache / f"{ryu:02d}.html"
                if f.exists() or parse_only:
                    continue
                for attempt in range(3):
                    try:
                        txt = fetch(sess, page, y, ryu)
                        break
                    except requests.RequestException as e:
                        logger.warning("  %s %d년 제%02d류 실패(%d회): %s", page, y, ryu, attempt + 1, e)
                        time.sleep(5 * (attempt + 1))
                else:
                    # 같은 자리에서 매번 끊기는 류가 있다(2011년 관세율표 제90류는 9006519000
                    # 행에서 서버 오류가 난다. 호 단위로 받아도 같다). 캐시하지 않고 넘어가며,
                    # build가 그 코드들을 주요세율보기로 채운다.
                    logger.error("  %s %d년 제%02d류를 받지 못해 건너뛴다", page, y, ryu)
                    continue
                f.write_text(txt, encoding="utf-8")
                meta.append((page, y, ryu, datetime.now(), len(txt)))
                n_req += 1
                time.sleep(delay)
            logger.info("%d년 %s 캐시 준비 (이번에 받은 누적 %d회)", y, page, n_req)
    return meta


def build(years, ryus):
    codes, rates, main_codes = [], [], []
    for y in years:
        for ryu in ryus:
            ft = PAGES["table"][1] / str(y) / f"{ryu:02d}.html"
            fm = PAGES["main"][1] / str(y) / f"{ryu:02d}.html"
            for f in (ft, fm):
                if f.exists() and ERROR_MARK in f.read_text(encoding="utf-8"):
                    logger.warning("  오류 안내문이 섞인 캐시: %s (지우고 다시 받을 것)", f)
            if ft.exists():
                c, r = parse_table(ft.read_text(encoding="utf-8"), y)
                codes += c
                rates += r
            if fm.exists():
                r, c = parse_main(fm.read_text(encoding="utf-8"), y)
                rates += r
                main_codes += c

    code = (pd.DataFrame(codes, columns=["year", "hs10", "name_ko"])
            .drop_duplicates(["year", "hs10"]).assign(source="table"))
    mc = pd.DataFrame(main_codes, columns=["year", "hs10", "name_ko"]).drop_duplicates(["year", "hs10"])
    fill = (mc.merge(code[["year", "hs10"]], how="left", indicator=True)
            .query("_merge == 'left_only'").drop(columns="_merge"))
    code = pd.concat([code, fill.assign(source="main")], ignore_index=True)
    rt = pd.DataFrame(rates, columns=["year", "hs10", "rate_cd", "rate_txt", "rate_nm", "period", "source"])

    # 관세율표 화면을 받지 못한 코드는 주요세율보기의 기본·WTO·아시아태평양 세율로 채운다.
    # 그 코드에 조정·할당·양허관세가 있었다면 빠진다(2011년 제90류는 앞뒤 해에 그런 세율이 없다).
    fk = set(zip(fill.year, fill.hs10))
    isfill = ((rt.source == "main") & ~rt.rate_cd.str.startswith("F")
              & pd.Series([(y, h) in fk for y, h in zip(rt.year, rt.hs10)], index=rt.index))
    rt.loc[isfill, "source"] = "fill"
    if len(fill):
        logger.warning("  관세율표 화면에 없어 주요세율보기로 채운 코드: %s",
                       fill.groupby("year").size().to_dict())
    rt["valid_from"] = [date(y, 1, 1) if p is None else date.fromisoformat(p[0])
                        for y, p in zip(rt.year, rt.period)]
    rt["valid_to"] = [date(y, 12, 31) if p is None or p[1] is None else date.fromisoformat(p[1])
                      for y, p in zip(rt.year, rt.period)]
    ad = rt.rate_txt.map(parse_rate)
    rt["adval"] = [a for a, _ in ad]
    rt["specific"] = [s for _, s in ad]

    names = rt.dropna(subset=["rate_nm"]).drop_duplicates("rate_cd")[["rate_cd", "rate_nm"]]
    names["source"] = "table"
    fta = pd.DataFrame([(k, v, "main") for k, v in MAIN_NAMES.items()], columns=names.columns)
    names = pd.concat([names, fta]).drop_duplicates("rate_cd")
    names.loc[names.rate_cd == "A", ["rate_nm", "source"]] = ["기본세율", "table"]
    if "A" not in set(names.rate_cd):
        names.loc[len(names)] = ["A", "기본세율", "table"]
    return code, rt, names


def verify(code: pd.DataFrame, rt: pd.DataFrame):
    """두 화면의 대조와 알려진 값 확인, 수입액 커버리지. 로그에 찍는 수치를 outputs/수집_검증.csv(item, key, value)에도 남긴다."""
    rows = []   # (item, key, value) — 논문·노트북이 대조할 수 있게 파일로 남긴다
    tb, mn = rt[rt.source == "table"], rt[rt.source == "main"]
    for y in sorted(code.year.unique()):
        ct = set(code[(code.year == y) & (code.source == "table")].hs10)
        cm = set(mn[mn.year == y].hs10)
        if cm:
            logger.info("  %d년 코드: 관세율표 %d / 주요세율보기 %d (한쪽에만 %d)",
                        y, len(ct), len(cm), len(ct ^ cm))
            rows += [("코드 집합 한쪽에만", y, len(ct ^ cm)), ("코드 관세율표", y, len(ct)), ("코드 주요세율보기", y, len(cm))]

    # 두 화면에 함께 나오는 기본세율(A)과 WTO 세율(C)이 같은가. 문구는 천 단위 쉼표가
    # 화면마다 달라('1218원'과 '1,218원') 숫자로 견준다.
    key = ["year", "hs10", "adval", "specific", "rate_txt"]
    for cd in ("A", "C"):
        a = tb[tb.rate_cd == cd].drop_duplicates(["year", "hs10"])[key]
        b = mn[mn.rate_cd == cd].drop_duplicates(["year", "hs10"])[key]
        m = a.merge(b, on=["year", "hs10"])
        if len(m):
            ok = (m.adval_x.fillna(-1) == m.adval_y.fillna(-1)) & (m.specific_x.fillna(-1) == m.specific_y.fillna(-1))
            logger.info("  %s 세율 두 화면 일치 %.2f%% (%d쌍)", cd, 100 * ok.mean(), len(m))
            rows += [(f"두 화면 일치율 {cd}", "전체", round(100 * ok.mean(), 2)), (f"두 화면 대조 쌍 {cd}", "전체", len(m))]
            if ok.mean() < 0.99:
                logger.warning("  어긋나는 예: %s", m[~ok].head(5).to_dict("records"))
    rt = rt[rt.source.isin(["table", "fill"]) | rt.rate_cd.str.startswith("F")]   # 적재 대상만 남긴다

    unread = rt[rt.adval.isna() & rt.specific.isna()]
    logger.info("  숫자로 못 읽은 세율 %d행 (%.3f%%) 예: %s", len(unread), 100 * len(unread) / max(len(rt), 1),
                unread.rate_txt.value_counts().head(5).to_dict())
    rows.append(("숫자로 못 읽은 세율 행", "전체", len(unread)))

    known = [  # (연도, 코드, 구분, 종가, 종량, 시작일)
        (2025, "0904210000", "W2", 270.0, 6210.0, None),
        (2009, "0910101000", "W2", 377.3, 931.0, None),
        (2025, "2103909050", "L", 45.0, None, None),
        (2025, "2103909050", "FCN1", 44.5, None, None),
        (2025, "2103909050", "FEU1", 5.6, None, date(2025, 1, 1)),
        (2025, "2103909050", "FEU1", 2.8, None, date(2025, 7, 1)),
        (2012, "8703231010", "C", 8.0, None, None),
        (2025, "0710807000", "C", 27.0, None, None),
    ]
    for y, h, cd, a, s, f in known:
        if y not in set(rt.year):
            continue
        hit = rt[(rt.year == y) & (rt.hs10 == h) & (rt.rate_cd == cd)]
        if f is not None:
            hit = hit[hit.valid_from == f]
        assert len(hit) == 1, f"{y} {h} {cd}: {len(hit)}행"
        r = hit.iloc[0]
        assert r["adval"] == a and (s is None or r["specific"] == s), f"{y} {h} {cd}: {r['rate_txt']}"
    logger.info("  알려진 값 확인 통과")
    rows.append(("알려진 값 확인", "건수", len([k for k in known if k[0] in set(rt.year)])))

    if DB_TRADE.exists():
        con = duckdb.connect(str(DB_TRADE), read_only=True)
        try:
            imp = con.execute("SELECT yyyymm//100 AS year, hs10, SUM(imp_dlr) AS v FROM fact_trade "
                              "GROUP BY 1, 2").df()
        finally:
            con.close()
        imp = imp[imp.year.isin(code.year.unique())]
        # 채움(fill) 코드까지 넣은 커버리지와, 관세율표 화면만으로 잰 커버리지(채움 전)를 함께 남긴다.
        # 2011년 제90류가 빠졌을 때 후자가 97.38%로 떨어져 결함이 드러났다.
        for label, sub in [("코드 있음", code), ("코드 있음(채움 전)", code[code.source == "table"])]:
            have = sub[["year", "hs10"]].drop_duplicates().assign(ok=1)
            m = imp.merge(have, on=["year", "hs10"], how="left")
            cov = m.assign(ok=m.ok.fillna(0)).groupby("year").apply(lambda g: (g.v * g.ok).sum() / g.v.sum())
            logger.info("  수입액 중 그해 세율표에 %s 몫: %s", label,
                        ", ".join(f"{y} {100 * c:.2f}%" for y, c in cov.items()))
            rows += [(f"수입액 커버리지 {label}", int(y), round(100 * c, 2)) for y, c in cov.items()]
    OUT_DIR.mkdir(exist_ok=True)
    pd.DataFrame(rows, columns=["item", "key", "value"]).to_csv(OUT_DIR / "수집_검증.csv", index=False, encoding="utf-8-sig")
    logger.info("  검증 수치 %d행 → %s", len(rows), OUT_DIR / "수집_검증.csv")


def load(code, rt, names, meta):
    """새 파일에 쓰고 바꿔 끼운다.

    DuckDB는 CREATE OR REPLACE로 표를 갈아도 파일이 줄지 않아 다시 적재할 때마다 커진다
    (한 번 다시 적재하자 27MB가 53MB가 됐다). 받은 기록(meta_fetch)만 옛 파일에서 옮겨 온다.
    """
    tmp = DB_OUT.with_name(DB_OUT.stem + ".tmp.duckdb")
    tmp.unlink(missing_ok=True)
    con = duckdb.connect(str(tmp))
    try:
        con.register("code_df", code)
        con.register("rt_df", rt.drop(columns=["period", "rate_nm"]))
        con.register("nm_df", names)
        con.execute("""
            CREATE OR REPLACE TABLE tariff_code AS
            SELECT CAST(year AS INTEGER) AS year, hs10, name_ko, source FROM code_df ORDER BY year, hs10""")
        con.execute("""
            CREATE OR REPLACE TABLE tariff_rate AS
            SELECT CAST(year AS INTEGER) AS year, hs10, rate_cd, rate_txt,
                   CAST(adval AS DOUBLE) AS adval, CAST(specific AS DOUBLE) AS specific,
                   CAST(valid_from AS DATE) AS valid_from, CAST(valid_to AS DATE) AS valid_to, source
            FROM rt_df ORDER BY year, hs10, rate_cd, valid_from""")
        con.execute("CREATE OR REPLACE TABLE dim_rate_cd AS SELECT * FROM nm_df ORDER BY rate_cd")
        con.execute("""CREATE TABLE meta_fetch
                       (page VARCHAR, year INTEGER, ryu INTEGER, fetched_at TIMESTAMP, bytes INTEGER)""")
        if DB_OUT.exists():
            con.execute(f"ATTACH '{DB_OUT.as_posix()}' AS old (READ_ONLY)")
            has = con.execute("SELECT COUNT(*) FROM duckdb_tables() "
                              "WHERE database_name = 'old' AND table_name = 'meta_fetch'").fetchone()[0]
            if has:
                con.execute("INSERT INTO meta_fetch SELECT * FROM old.meta_fetch")
            con.execute("DETACH old")
        if meta:
            con.executemany("INSERT INTO meta_fetch VALUES (?, ?, ?, ?, ?)", meta)
        con.execute("COMMENT ON TABLE tariff_rate IS '관세법령정보포털 연도별 관세율표(기본·탄력·양허)와 "
                    "주요세율보기(FTA 일곱 상대). FTA는 적용 가능 세율이지 납부 세율이 아니다. 포털 고지상 법적 효력 없음'")
        for t in ("tariff_code", "tariff_rate", "dim_rate_cd"):
            logger.info("  %s %d행", t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])
    finally:
        con.close()
    os.replace(tmp, DB_OUT)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--years", type=int, nargs="+", default=list(YEARS))
    ap.add_argument("--ryu", type=int, nargs="+", default=list(RYU), help="시험용: 일부 류만")
    ap.add_argument("--pages", nargs="+", default=list(PAGES), choices=list(PAGES))
    ap.add_argument("--delay", type=float, default=0.7, help="요청 간격(초)")
    ap.add_argument("--parse-only", action="store_true", help="받지 않고 캐시만 적재")
    ap.add_argument("--verify-only", action="store_true", help="캐시를 파싱해 검증만 하고 적재하지 않는다(수집_검증.csv 갱신)")
    args = ap.parse_args()
    if args.verify_only:
        args.parse_only = True

    meta = collect(args.years, args.ryu, args.pages, args.delay, args.parse_only)
    code, rt, names = build(args.years, args.ryu)
    logger.info("파싱: 코드 %d, 세율 %d행", len(code), len(rt))
    verify(code, rt)
    if args.verify_only:
        logger.info("검증만 하고 적재는 하지 않는다"); return
    # 주요세율보기의 기본·WTO·아시아태평양 열은 관세율표와 겹치므로 대조에만 쓰고 FTA 열만 싣는다.
    # 관세율표 화면을 받지 못해 채운 코드(fill)는 예외다.
    rt = rt[rt.source.isin(["table", "fill"]) | rt.rate_cd.str.startswith("F")]
    load(code, rt, names, meta)
    logger.info("완료: %s", DB_OUT)


if __name__ == "__main__":
    main()


## 2. 실행세율 — 관세법 제50조의 우선순위를 코드×연도에 적용

본문 IV장, 부록 A2.1. `--load`로 DB에 적재하고 법령 대조 표본 CSV를 남긴다.

In [ ]:
# ===== 파일: scripts/02_build_applied_rate.py (그대로 옮김, 13,746바이트) =====
"""
02_build_applied_rate.py — 연도×HS10의 실행세율(applicable rate)을 만든다.

근거: 관세법 제50조(세율 적용의 우선순위), FTA 관세특례법 제5조, 양허관세 규정 제6조.
조문 원문과 세율 구분 코드의 대응은 docs/실행세율_법령근거.md.

규칙(연구 문서 IV.1절):
  기본   = A(기본세율). 잠정세율은 자료에 없다.
  3순위  = P3(할당관세, 수입전량)가 있으면 P3, 아니면 L(조정관세)가 있으면 L, 아니면 기본.
           P1(할당관세 물량이내 추천)은 추천이 있어야 하므로 표시만 한다.
  2순위  = C(WTO 협정세율)·F(국제협력관세)는 3순위보다 낮을 때만 우선(제50조③ 본문).
           W2(농림축산물 양허관세, 별표 1의 나)는 기본세율보다 높아도 우선(제50조③ 단서)하되
           할당관세(P3)가 있으면 그것을 따른다. W1(추천)은 추천이 있어야 하므로 표시만 한다.
  1순위  = I(덤핑방지)·T1·T2(특별긴급)는 원산지·물량 조건이 붙는 추가 관세라 표시만 한다.
  협정   = FTA 일곱 상대(FCN1·FEU1·FUS1·FAS1·FIN1·FVN1·FCA1)와 APTA(E1·E2·E3)는
           무협정 적용세율보다 낮을 때만 적용(특례법 제5조①, 제50조③).
  연중 변경 = 구간별 세율을 그해 유효 일수로 가중평균한다(rate_*), 1월 1일 세율도 둔다(*_jan).
  종량 하한 = 선택 세율("N% 또는 M원")이 있는 규정에서 M/N(원/kg). 격차 변수에는 종가세율만 쓴다.

산출:
  outputs/fct_applied_rate.parquet (연도×HS10)
  outputs/dim_origin_regime.csv     (원산지→협정)
  data/processed/kcstariff.duckdb 의 fct_applied_rate, dim_origin_regime (--load 를 주면 적재)
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

import os
ROOT = Path(__file__).resolve().parent.parent
TARIFF = ROOT / "data" / "processed" / "kcstariff.duckdb"
KCS = Path(os.environ.get("KCSDB2_PATH", ROOT.parent / "KCSDB2" / "data" / "processed" / "kcsdb.duckdb"))   # 있으면 커버리지 검증
OUT = ROOT / "outputs"

FTA = {"FCN1": "cn", "FEU1": "eu", "FUS1": "us", "FAS1": "asean", "FIN1": "in", "FVN1": "vn", "FCA1": "ca"}

# 원산지(관세청 stat_cd = ISO2) → 협정. (regime, from_year, to_year)
EU27 = ["AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE"]
ASEAN = ["BN", "KH", "ID", "LA", "MY", "MM", "PH", "SG", "TH", "VN"]
APTA_E1 = ["CN", "IN", "LK", "MN"]      # 일반 협정세율
ORIGIN_REGIME = (
    [("CN", "FCN1", 2015, 9999), ("IN", "FIN1", 2010, 9999), ("US", "FUS1", 2012, 9999), ("VN", "FVN1", 2015, 9999), ("CA", "FCA1", 2015, 9999)]
    + [(c, "FEU1", 2011, 9999) for c in EU27 if c != "HR"] + [("HR", "FEU1", 2013, 9999)] + [("GB", "FEU1", 2011, 2020)]
    + [(c, "FAS1", 2008, 9999) for c in ASEAN]
    + [(c, "E1", 2007, 9999) for c in APTA_E1] + [("BD", "E2", 2007, 9999), ("LA", "E3", 2007, 9999)]
)


def load_rates() -> pd.DataFrame:
    con = duckdb.connect()
    con.execute(f"ATTACH '{TARIFF.as_posix()}' AS tr (READ_ONLY)")
    df = con.sql("SELECT year, hs10, rate_cd, rate_txt, adval, specific, valid_from, valid_to FROM tr.tariff_rate").df()
    con.close()
    df["valid_from"] = pd.to_datetime(df.valid_from)
    df["valid_to"] = pd.to_datetime(df.valid_to)
    return df


def weight_by_days(df: pd.DataFrame) -> pd.DataFrame:
    """(year, hs10, rate_cd)마다 유효 일수 가중 종가세율, 1월 1일 세율, 종량세(원/kg)를 만든다."""
    ystart = pd.to_datetime(df.year.astype(str) + "-01-01")
    yend = pd.to_datetime(df.year.astype(str) + "-12-31")
    vf = df.valid_from.fillna(ystart).clip(lower=ystart)
    vt = df.valid_to.fillna(yend).clip(upper=yend)
    df = df.assign(days=(vt - vf).dt.days.clip(lower=0) + 1, vf=vf)
    has = df.adval.notna()
    df["wnum"] = np.where(has, df.adval.fillna(0) * df.days, 0.0)
    df["wden"] = np.where(has, df.days, 0)
    keys = ["year", "hs10", "rate_cd"]
    g = df.groupby(keys, sort=False)
    out = g.agg(wnum=("wnum", "sum"), wden=("wden", "sum"), specific=("specific", "max"), txt=("rate_txt", "first")).reset_index()
    out["rate"] = np.where(out.wden > 0, out.wnum / out.wden.replace(0, np.nan), np.nan)
    # 1월 1일 세율: 시작일이 가장 이른 행(종가 있는 것 우선)
    first = df[has].sort_values(keys + ["vf"]).drop_duplicates(keys)[keys + ["adval"]].rename(columns={"adval": "rate_jan"})
    out = out.merge(first, on=keys, how="left")
    return out.drop(columns=["wnum", "wden"])


def build(df: pd.DataFrame) -> pd.DataFrame:
    w = weight_by_days(df)
    rate = w.pivot(index=["year", "hs10"], columns="rate_cd", values="rate")
    jan = w.pivot(index=["year", "hs10"], columns="rate_cd", values="rate_jan")
    spec = w.pivot(index=["year", "hs10"], columns="rate_cd", values="specific")
    txt = w.pivot(index=["year", "hs10"], columns="rate_cd", values="txt")
    t = pd.DataFrame(index=rate.index)
    for cd in ["A", "C", "F", "L", "P3", "W2", "W1", "E1", "E2", "E3"] + list(FTA):
        t[f"r_{cd}"] = rate.get(cd)
        t[f"j_{cd}"] = jan.get(cd)
    for cd in ["A", "C", "W2"]:
        t[f"spec_{cd}"] = spec.get(cd)
        t[f"txt_{cd}"] = txt.get(cd)
    for cd in ["I", "T1", "T2", "P1", "W1", "D", "G1", "G2"]:
        t[f"has_{cd}"] = rate.get(cd).notna() if cd in rate.columns else False

    # ---- 무협정 적용세율(MFN, 제50조) ----
    base = t.r_A
    domestic = t.r_P3.where(t.r_P3.notna(), t.r_L.where(t.r_L.notna(), base))
    regime = np.where(t.r_P3.notna(), "P3", np.where(t.r_L.notna(), "L", "A"))
    tier2 = t[["r_C", "r_F"]].min(axis=1)
    use2 = tier2.notna() & (domestic.isna() | (tier2 < domestic))
    mfn = domestic.where(~use2, tier2)
    regime = np.where(use2, np.where(t.r_C.notna() & (t.r_C <= t.r_F.fillna(np.inf)), "C", "F"), regime)
    # 양허 농림축산물(W2): 기본세율에 우선. 할당관세(P3)가 있으면 그것.
    usew = t.r_W2.notna() & t.r_P3.isna()
    mfn = mfn.where(~usew, t.r_W2)
    regime = np.where(usew, "W2", regime)
    t["mfn"] = mfn
    t["mfn_regime"] = regime
    # 1월 1일 기준 MFN(같은 규칙)
    domestic_j = t.j_P3.where(t.j_P3.notna(), t.j_L.where(t.j_L.notna(), t.j_A))
    tier2_j = t[["j_C", "j_F"]].min(axis=1)
    use2_j = tier2_j.notna() & (domestic_j.isna() | (tier2_j < domestic_j))
    mfn_j = domestic_j.where(~use2_j, tier2_j)
    t["mfn_jan"] = mfn_j.where(~usew, t.j_W2)

    # ---- 협정·APTA 적용세율: 무협정보다 낮을 때만 ----
    for cd, name in FTA.items():
        r = t[f"r_{cd}"]
        t[f"applied_{name}"] = np.where(r.notna() & (r < t.mfn), r, t.mfn)
    for cd, name in [("E1", "apta"), ("E2", "apta_bd"), ("E3", "apta_la")]:
        r = t[f"r_{cd}"]
        t[f"applied_{name}"] = np.where(r.notna() & (r < t.mfn), r, t.mfn)
    # 중국·인도는 FTA와 APTA 중 낮은 쪽, 베트남·라오스는 FTA와 아세안 중 낮은 쪽
    t["applied_cn"] = np.minimum(t.applied_cn, t.applied_apta)
    t["applied_in"] = np.minimum(t.applied_in, t.applied_apta)
    t["applied_vn"] = np.minimum(t.applied_vn, t.applied_asean)
    t["applied_la"] = np.minimum(t.applied_asean, t.applied_apta_la)

    # ---- 종량 하한(원/kg): MFN 규정이 선택 세율이면 종량/종가 ----
    spec_used = np.where(t.mfn_regime == "W2", t.spec_W2, np.where(t.mfn_regime == "C", t.spec_C, np.where(t.mfn_regime == "A", t.spec_A, np.nan)))
    spec_used = np.where(spec_used > 0, spec_used, np.nan)   # "N% 또는 0원"은 종량 대안이 아니다
    t["specific_won_kg"] = spec_used
    t["floor_won_kg"] = np.where((t.mfn > 0) & pd.notna(spec_used), spec_used / (t.mfn / 100.0), np.nan)
    t = t.reset_index()
    # 세율 미확정: 종량세만 있는 코드(영화필름), 두 별표에 함께 오른 코드(인삼 기타),
    # 관세화(2015) 전의 쌀(수입이 시장접근물량으로 제한되어 물량 밖 세율이 없다)
    t["undetermined_reason"] = ""
    t.loc[t.mfn.isna(), "undetermined_reason"] = "specific_only"
    t.loc[t.hs10 == "1211209900", "undetermined_reason"] = "two_annexes"
    t.loc[t.hs10.str.startswith("1006") & (t.year < 2015), "undetermined_reason"] = "rice_pre_tariffication"
    t["rate_undetermined"] = t.undetermined_reason != ""
    return t


def validate(t: pd.DataFrame) -> None:
    known = [  # (year, hs10, column, expected)
        (2025, "2103909050", "mfn", 45.0), (2025, "2103909050", "applied_cn", 44.5),
        (2025, "0904210000", "mfn", 270.0), (2025, "0904210000", "applied_cn", 270.0),
        (2025, "0710807000", "mfn", 27.0),
        (2025, "0910111000", "mfn", 377.3),
        (2025, "0710809090", "mfn", 27.0),
        (2023, "2103909090", "applied_eu", (11.2 * 181 + 8.4 * 184) / 365),
    ]
    bad = []
    for y, h, col, exp in known:
        row = t[(t.year == y) & (t.hs10 == h)]
        got = float(row[col].iloc[0]) if len(row) else np.nan
        if not np.isclose(got, exp, atol=0.05):
            bad.append((y, h, col, exp, got))
    if bad:
        for b in bad:
            print("  FAIL", b)
        raise SystemExit("알려진 값과 어긋난다")
    print(f"검증: 알려진 값 {len(known)}건 통과")
    if not KCS.exists():
        print("수입액 커버리지 검증은 KCSDB2가 없어 건너뛴다"); return
    con = duckdb.connect()
    con.execute(f"ATTACH '{KCS.as_posix()}' AS s (READ_ONLY)")
    con.register("t", t[["year", "hs10", "mfn"]])
    cov = con.sql("""
        WITH imp AS (SELECT yyyymm//100 yr, hs10, sum(imp_dlr) v FROM s.fact_trade GROUP BY 1,2)
        SELECT imp.yr, round(100.0*sum(CASE WHEN t.mfn IS NOT NULL THEN v ELSE 0 END)/sum(v),2) pct_rate,
               round(100.0*sum(CASE WHEN t.hs10 IS NOT NULL THEN v ELSE 0 END)/sum(v),2) pct_code
        FROM imp LEFT JOIN t ON t.year=imp.yr AND t.hs10=imp.hs10 GROUP BY 1 ORDER BY 1""").df()
    print("수입액 커버리지(%): 코드 있음 / 종가 실행세율 있음")
    print(cov.to_string(index=False))
    cov.to_csv(OUT / "실행세율_커버리지.csv", index=False, encoding="utf-8-sig")
    con.close()


def write_legal_sample(t: pd.DataFrame) -> None:
    """법령 대조 표본(2025년): 파일럿 코드 전부 + 별표 43개 항목의 호마다 2개 + 무작위 100개. 사람이 법령과 대조해 채우는 표."""
    import re
    rule = OUT / "분류기준_규칙_별표.csv"   # 연구 저장소의 표. 없으면 무작위 표본만 만든다
    hs4 = set()
    if rule.exists():
        r = pd.read_csv(rule, dtype=str)
        for c in ["충족 시 호", "반대편 호"]:
            for v in r[c].dropna():
                hs4 |= {h[:4] for h in re.findall(r"\d{4}", v)}
    # 파일럿 코드는 전부, 별표의 호마다 무작위 2개
    codes = {"0904210000", "0904220000", "0710807000", "2103909050", "2103909090", "0910111000", "0910112000",
             "0910113000", "0910123000", "0710809090", "0813400000", "0810909000", "0811909000", "1211209900"}
    y = t[t.year == 2025]
    pilot = y[y.hs10.isin(codes)].assign(sample="pilot")
    head = y[y.hs10.str[:4].isin(hs4) & ~y.hs10.isin(codes)].groupby(y.hs10.str[:4], group_keys=False).apply(lambda g: g.sample(min(2, len(g)), random_state=1)).assign(sample="heading")
    rest = y[~y.hs10.isin(codes) & ~y.hs10.isin(head.hs10)].sample(100, random_state=20260912).assign(sample="random")
    pair = pd.concat([pilot, head])
    cols = ["sample", "year", "hs10", "r_A", "r_C", "r_W2", "r_L", "r_P3", "mfn", "mfn_regime", "specific_won_kg", "floor_won_kg",
            "applied_cn", "applied_us", "applied_eu", "applied_asean", "applied_vn", "has_W1", "has_P1", "has_I", "has_T1", "has_T2", "rate_undetermined"]
    out = pd.concat([pair, rest])[cols].sort_values(["sample", "hs10"])
    out["법령_확인세율"] = ""
    out["확인_출처"] = ""
    out["비고"] = ""
    out.to_csv(OUT / "실행세율_법령대조.csv", index=False, encoding="utf-8-sig")
    print(f"법령 대조 표본: 파일럿 {len(pilot)} + 별표 호별 {len(head)} + 무작위 {len(rest)} → outputs/실행세율_법령대조.csv")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--load", action="store_true", help="kcstariff.duckdb 에 적재")
    args = ap.parse_args()
    df = load_rates()
    t = build(df)
    validate(t)
    OUT.mkdir(exist_ok=True)
    t.to_parquet(OUT / "fct_applied_rate.parquet", index=False)
    dim = pd.DataFrame(ORIGIN_REGIME, columns=["stat_cd", "regime", "from_year", "to_year"])
    dim.to_csv(OUT / "dim_origin_regime.csv", index=False, encoding="utf-8-sig")
    print(f"fct_applied_rate {len(t):,}행, 미확정 {int(t.rate_undetermined.sum()):,}행 {t[t.rate_undetermined].undetermined_reason.value_counts().to_dict()}, 종량 하한 있음 {int(t.floor_won_kg.notna().sum()):,}행")
    print("MFN 규정 분포:", t.mfn_regime.value_counts().to_dict())
    write_legal_sample(t)
    if args.load:
        con = duckdb.connect(str(TARIFF))
        con.execute(f"CREATE OR REPLACE TABLE fct_applied_rate AS SELECT * FROM read_parquet('{(OUT / 'fct_applied_rate.parquet').as_posix()}')")
        con.execute(f"CREATE OR REPLACE TABLE dim_origin_regime AS SELECT * FROM read_csv_auto('{(OUT / 'dim_origin_regime.csv').as_posix()}')")
        con.close()
        print("적재 완료")


if __name__ == "__main__":
    main()


## 3. 양허관세 규정 별표 1(가·나) — XLSX를 코드별 표로

부록 A2.2. 2026-09-13 복원 스크립트. 옛 산출과 10,222행 전부 같다.

In [ ]:
# ===== 파일: research/scripts/25_parse_concession_annex.py (그대로 옮김, 4,564바이트) =====
"""
25_parse_concession_annex.py — 세계무역기구협정 등에 의한 양허관세 규정 별표 1(가·나) XLSX → 코드별 표(자료 논문 부록 A2.2).

2026-09-12에 대화형으로 만든 outputs/양허관세_별표1_2025.csv 를 2026-09-13에 스크립트로 복원했다. 복원본은 옛 CSV와 10,222행 전부 같다
(열 12개 값 비교, 부분 양허 155·종량 대안 92·두 별표 겹침 1211209900).

입력: data/external/양허관세_규정_별표1/ 의 XLSX 둘(법령센터 ZIP 안, 2024-12-31 개정본).
  가: 열 0~4 = 호(4자리)·소호(2)·세분(4)·품명·세율. 세 칸이 다 있는 행이 10단위 코드 행이고, 품명이 「-」로 시작하는 행은 바로 위 코드의
      세부 품목에만 양허한 「부분 양허」(품명 : 세율)다. 호·소호만 있는 행은 제목이라 버린다.
  나: 열 0~6 = 앞 6자리·2·2·품명·시장접근물량·물량 이내 세율·물량 초과 세율. 뒤 두 칸이 다 있는 행이 10단위 코드 행이다.
세율 문구는 그대로 두고(rate_txt) 종가(%)·종량(원/kg)·「양자 중 고액」 여부를 따로 읽는다. 가·나 양쪽에 있는 코드는 미확정으로 표시한다.
"""
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
EXT = ROOT / "data" / "external" / "양허관세_규정_별표1"
GA = EXT / "1-1. 세계무역기구협정 등에 의한 양허관세 규정 별표 1의 가.xlsx"
NA = EXT / "1-2. 세계무역기구협정 등에 의한 양허관세 규정 별표 1의 나.xlsx"
OUT = ROOT / "outputs" / "양허관세_별표1_2025.csv"


def cell(v, code=False) -> str:
    """code=True: 코드 칸(숫자 셀 1000.0 → '1000'). 세율 칸은 숫자 셀을 '18.0' 꼴 그대로 둔다."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    s = str(v).strip()
    return s[:-2] if (code and re.fullmatch(r"\d+\.0", s)) else s


def parse_rate(txt: str):
    m = re.search(r"(\d+(?:\.\d+)?)\s*%", txt)
    adval = float(m.group(1)) if m else (float(txt) if re.fullmatch(r"\d+(\.\d+)?", txt) else np.nan)
    m2 = re.search(r"([\d,]+(?:\.\d+)?)\s*원", txt)
    spec = float(m2.group(1).replace(",", "")) if m2 else np.nan
    return adval, spec, ("양자 중 고액" in txt)


def parse_ga() -> list[dict]:
    rows = []
    for r in pd.read_excel(GA, header=None, dtype=object).itertuples(index=False):
        c0, c1, c2 = (cell(v, code=True) for v in r[:3]); name, rate = (cell(v) for v in r[3:5])
        if re.fullmatch(r"\d{4}", c0) and re.fullmatch(r"\d{2}", c1) and re.fullmatch(r"\d{4}", c2):
            rows.append(dict(byeolpyo="1의 가", hs10=c0 + c1 + c2, name_ko=name, trq="", in_quota="", rate_txt=rate, partial=[]))
        elif name.startswith("-") and rows:
            rows[-1]["partial"].append(f"{name} : {rate}" if rate else name)
    return rows


def parse_na() -> list[dict]:
    rows = []
    for r in pd.read_excel(NA, header=None, dtype=object).itertuples(index=False):
        c0, c1, c2 = (cell(v, code=True) for v in r[:3]); name, trq, inq, rate = (cell(v) for v in r[3:7])
        if re.fullmatch(r"\d{6}", c0) and re.fullmatch(r"\d{2}", c1) and re.fullmatch(r"\d{2}", c2):
            rows.append(dict(byeolpyo="1의 나", hs10=c0 + c1 + c2, name_ko=name, trq=trq, in_quota=inq, rate_txt=rate, partial=[]))
    return rows


def build() -> pd.DataFrame:
    out = []
    for d in parse_ga() + parse_na():
        adval, spec, hi = parse_rate(d["rate_txt"])
        out.append(dict(byeolpyo=d["byeolpyo"], hs10=d["hs10"], name_ko=d["name_ko"], trq=d["trq"], in_quota=d["in_quota"], rate_txt=d["rate_txt"],
                        adval=adval, specific_won_kg=spec, higher_of=hi, partial=" | ".join(d["partial"])))
    df = pd.DataFrame(out)
    both = set(df[df.byeolpyo == "1의 가"].hs10) & set(df[df.byeolpyo == "1의 나"].hs10)
    df["rate_undetermined"] = np.where(df.hs10.isin(both), "Y", "")
    df["undetermined_reason"] = np.where(df.hs10.isin(both), "별표 1의 가와 나에 모두 있어 물품 정의에 따라 세율이 갈림", "")
    return df


def main() -> None:
    df = build()
    df.to_csv(OUT, index=False, encoding="utf-8-sig")
    print(f"{OUT.name}: {len(df):,}행 {df.byeolpyo.value_counts().to_dict()}, 부분 양허 {int((df.partial != '').sum())}, 종량 대안 코드 {df[df.higher_of].hs10.nunique()}, 두 별표 겹침 {int((df.rate_undetermined == 'Y').sum())}행")


if __name__ == "__main__":
    main()


## 4. 품목분류 적용기준 규칙 별표 — PDF 텍스트 추출

부록 A2.3. 법령센터의 `flDownload.do?flSeq=111179395`(PDF 17쪽)를 받은 뒤 아래처럼 텍스트를 뽑았다. 추출 텍스트에 낱말 사이 공백이 빠져 있어 43개 항목의 번호·품명·수치 기준·호는 손으로 `분류기준_규칙_별표.csv`에 옮기고 원문과 대조했다.

In [ ]:
import fitz  # pymupdf
from pathlib import Path

pdf = Path(r"research/data/external/품목분류_적용기준_규칙/별표_품목분류의_적용기준_2022.pdf")
doc = fitz.open(str(pdf))
text = "\n".join(page.get_text() for page in doc)
pdf.with_suffix(".txt").write_text(text, encoding="utf-8")
print(len(doc), "쪽,", len(text), "자")


## 5. 유통이력관리 고시 별표 1 — HWP 5.0·HWPX 텍스트 읽기

부록 A2.4. 판본 조각 페이지의 `flDownload.do?flSeq=<번호>` 링크를 `requests`로 받아(파일 이름에 `flSeq`까지 붙여 저장) 머리 바이트로 형식을 판별한 뒤, 아래 두 함수로 문단 텍스트를 뽑았다. 표(판본별 1,275행·이관 후 310행, 지정 구간 331·180, 코드 요약 135, 통합 구간 237)는 이 텍스트에서 대화형으로 만들었고 그 규칙(코드의 점·붙임표 정리, 지정기간 끝이 빈 판본, 판본마다 바뀌는 항목 이름, 세 고시의 병합)은 부록 A2.4에 있다.

In [ ]:
import olefile, zlib, struct, zipfile, re
from pathlib import Path


def file_kind(path: Path) -> str:
    head = path.read_bytes()[:4]
    return "hwp" if head == b"\xd0\xcf\x11\xe0" else ("hwpx" if head[:2] == b"PK" else ("pdf" if head == b"%PDF" else "unknown"))


def hwp_text(path):
    """HWP 5.0(OLE): FileHeader 36번째 바이트의 최하위 비트가 압축 여부, BodyText/Section<n>의 PARA_TEXT(태그 67) 레코드가 문단."""
    ole = olefile.OleFileIO(str(path))
    compressed = bool(ole.openstream("FileHeader").read()[36] & 1)
    out = []
    for entry in sorted(ole.listdir()):
        if entry[0] != "BodyText":
            continue
        data = ole.openstream(entry).read()
        if compressed:
            data = zlib.decompress(data, -15)
        i = 0
        while i + 4 <= len(data):
            head = struct.unpack("<I", data[i:i + 4])[0]
            tag, size = head & 0x3FF, (head >> 20) & 0xFFF
            i += 4
            if size == 0xFFF:
                size = struct.unpack("<I", data[i:i + 4])[0]
                i += 4
            if tag == 67:                       # PARA_TEXT
                out.append(data[i:i + size].decode("utf-16le", errors="ignore"))
            i += size
    return "\n".join(re.sub(r"[\x00-\x1f]", " ", t) for t in out)


def hwpx_text(path):
    """HWPX(ZIP): Contents/section<n>.xml 의 <hp:t>…</hp:t> 문자열을 순서대로."""
    z = zipfile.ZipFile(path)
    out = []
    for name in sorted(z.namelist()):
        if re.match(r"Contents/section\d+\.xml", name):
            xml = z.read(name).decode("utf-8")
            out.append(" ".join(re.sub(r"<[^>]+>", "", t) for t in re.findall(r"<hp:t[^>]*>(.*?)</hp:t>", xml, re.S)))
    return "\n".join(out)


if __name__ == "__main__":
    base = Path(r"research/data/external/유통이력관리_고시_별표1")
    for p in sorted(base.glob("*.hwp*")):
        kind = file_kind(p)
        text = hwp_text(p) if kind == "hwp" else hwpx_text(p) if kind == "hwpx" else ""
        print(p.name, kind, len(text), "자")


## 6. 사전세액심사 대상물품 — 페이지 컨텍스트 `fetch`

부록 A1.4. 포털의 조회 화면 `openULS0105013Q.do`를 브라우저로 연 뒤 같은 출처에서 데이터 요청 `retrieveBtaaTrgtCmdt.do`를 보낸다(브라우저 밖 `requests`에는 빈 응답). 2016년 1월부터 2026년 9월까지 매월 1일 기준 129회. 결과 JSON을 `사전세액심사_대상_월별_2016_2026.csv`(ym·hsSgn)와 코드별 이력 CSV로 옮기는 것은 대화형이었다.

```javascript
const out = [];
for (let y = 2016; y <= 2026; y++) for (let m = 1; m <= 12; m++) {
  const d = `${y}${String(m).padStart(2, "0")}01`;
  const r = await fetch("/clip/lworsrch/retrieveBtaaTrgtCmdt.do",
    { method: "POST", headers: { "Content-Type": "application/x-www-form-urlencoded" }, body: "aplyDt=" + d });
  const j = await r.json();
  for (const x of [...(j.items1 || []), ...(j.items2 || [])])
    for (const s of ["1", "2"])
      if (x["hsSgn" + s]) out.push([d, x["hsSgn" + s], (x["hsSgnNm" + s] || "").trim(), x["aplyStrtDt" + s], x["btaaSelcBaseCd" + s], x["btaaSelcBaseNm" + s]]);
}
JSON.stringify(out);
```

## 7. 품목 상세 화면 — 연구 대상 코드의 그 밖 FTA 세율과 부가 정보

부록 A1.3. 코드×연도마다 한 번, 285코드 × 2012~2026년 = 4,275쪽.

In [ ]:
# ===== 파일: research/scripts/17_fetch_item_detail.py (그대로 옮김, 5,947바이트) =====
"""
17_fetch_item_detail.py — 관세법령정보포털 품목 상세(openULS0201007Q)에서 코드 쌍에 든 코드의
연도별 세율 전부(그 밖 FTA 포함)와 부가 정보(사전세액심사 대상 표시, 유통이력 신고대상 지정기간,
분류사례·원산지결정기준 건수)를 받는다. 상세 화면은 2012년부터만 세율을 준다.

입력: research/outputs/코드쌍_코드목록.txt (한 줄에 HS10 하나)
캐시: data/raw/clip_item_detail/<year>/<hs10>.html
산출: research/outputs/품목상세_세율_2012_2026.csv (연도×코드×구분기호, 긴 형태)
      research/outputs/품목상세_부가정보_2012_2026.csv
세율 문구에 " / "가 있으면 연중 구간이 여럿이다(구간 날짜는 화면에 없다). 첫 구간과 마지막 구간을 따로 둔다.
"""
from __future__ import annotations

import argparse
import html
import re
import time
from pathlib import Path

import pandas as pd
import requests

ROOT = Path(__file__).resolve().parents[1]                      # KCSTARIFF/research
RAW = ROOT / "data" / "raw" / "clip_item_detail"
OUT = ROOT / "outputs"
CODES = OUT / "코드쌍_코드목록.txt"
BASE = "https://unipass.customs.go.kr/clip/hsinfosrch/openULS0201007Q.do"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)", "Referer": "https://unipass.customs.go.kr/clip/index.do"}
YEARS = range(2012, 2027)
ERROR_MARK = "프로그램 오류발생"


def strip(x: str) -> str:
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", x))).strip()


def fetch(sess: requests.Session, code: str, year: int) -> str:
    r = sess.post(BASE, data={"searchVal": code, "aplyYy": str(year)}, timeout=60)
    r.raise_for_status()
    if ERROR_MARK in r.text:
        raise requests.RequestException("오류 안내문")
    return r.text


def parse_rate(txt: str) -> tuple[float | None, float | None]:
    """'270% 또는 6,210원' → (270.0, 6210.0); '240원' → (None, 240.0); '0%' → (0.0, None)."""
    a = re.search(r"([\d.]+)\s*%", txt)
    sp = re.search(r"([\d,]+(?:\.\d+)?)\s*원", txt)
    return (float(a.group(1)) if a else None, float(sp.group(1).replace(",", "")) if sp else None)


def parse(page: str, code: str, year: int) -> tuple[list[dict], dict]:
    rows = []
    i = page.find("세율적용 우선순위")
    j = page.find("내국세", i) if i > 0 else -1
    seg = page[i:j] if i > 0 and j > i else ""
    for tr in re.findall(r"<tr[^>]*>(.*?)</tr>", seg, re.S):
        c = [strip(x) for x in re.findall(r"<t[dh][^>]*>(.*?)</t[dh]>", tr, re.S)]
        if len(c) >= 3 and re.fullmatch(r"[A-Z][A-Z0-9]*", c[0]):
            segs = [s.strip() for s in c[1].split(" / ")]
            a1, s1 = parse_rate(segs[0])
            aN, sN = parse_rate(segs[-1])
            rows.append(dict(year=year, hs10=code, rate_cd=c[0], rate_name=c[2], rate_txt=c[1], n_seg=len(segs),
                             adval_first=a1, specific_first=s1, adval_last=aN, specific_last=sN))
    text = strip(re.sub(r"<script.*?</script>|<style.*?</style>", "", page, flags=re.S))
    m_yu = re.search(r"유통이력 신고대상물품명, 지정기간, 통보기관 (.*?) 연관정보", text)
    yu = m_yu.group(1).strip() if m_yu else ""
    if yu.startswith("There were no results"):
        yu = ""
    cnt = {k: int(m.group(1)) if (m := re.search(k + r"\s*(\d+)건", text)) else None
           for k in ["분류사례", "평가사례", "원산지결정기준", "판례·결정례"]}
    extra = dict(year=year, hs10=code, pre_assessment="사전세액대상물품" in text, deposit_price="담보기준가격" in text,
                 distribution_history=yu, n_class_case=cnt["분류사례"], n_valuation_case=cnt["평가사례"],
                 n_origin_rule=cnt["원산지결정기준"], n_ruling=cnt["판례·결정례"], n_rates=len(rows))
    return rows, extra


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--delay", type=float, default=0.5)
    ap.add_argument("--parse-only", action="store_true")
    ap.add_argument("--codes", nargs="*", help="지정하면 이 코드만")
    args = ap.parse_args()
    codes = args.codes or [c.strip() for c in CODES.read_text().splitlines() if c.strip()]
    sess = requests.Session()
    sess.headers.update(HEADERS)
    rates, extras, failed = [], [], []
    n = 0
    for code in codes:
        for year in YEARS:
            f = RAW / str(year) / f"{code}.html"
            if f.exists():
                page = f.read_text(encoding="utf-8")
            elif args.parse_only:
                continue
            else:
                try:
                    page = fetch(sess, code, year)
                except Exception as e:  # noqa: BLE001
                    failed.append((code, year, str(e)[:80]))
                    time.sleep(args.delay * 4)
                    continue
                f.parent.mkdir(parents=True, exist_ok=True)
                f.write_text(page, encoding="utf-8")
                time.sleep(args.delay)
            r, x = parse(page, code, year)
            rates.extend(r)
            extras.append(x)
        n += 1
        if n % 20 == 0:
            print(f"{n}/{len(codes)} 코드, 세율 행 {len(rates):,}, 실패 {len(failed)}", flush=True)
    pd.DataFrame(rates).to_csv(OUT / "품목상세_세율_2012_2026.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(extras).to_csv(OUT / "품목상세_부가정보_2012_2026.csv", index=False, encoding="utf-8-sig")
    print(f"완료: 코드 {len(codes)}, 세율 행 {len(rates):,}, 부가정보 {len(extras):,}, 실패 {len(failed)}")
    if failed:
        pd.DataFrame(failed, columns=["hs10", "year", "err"]).to_csv(OUT / "품목상세_실패.csv", index=False, encoding="utf-8-sig")
        print(failed[:10])


if __name__ == "__main__":
    main()


## 8. UN Comtrade 공개 미리보기 API — 미러 통계

부록 A3.2. 호출당 500행 상한과 429 대응.

In [ ]:
# ===== 파일: research/scripts/21_fetch_comtrade_mirror.py (그대로 옮김, 6,143바이트) =====
"""
21_fetch_comtrade_mirror.py — UN Comtrade 공개 미리보기 API(키 없음, 호출당 500행)로 주요 원산지가 보고한
대한국 HS6 수출(미러 통계)을 받는다. 대상 HS6는 확정 코드 쌍과 처리군 코드의 앞 6자리이며, 그해 HS 판본의
옛 코드(dim_hs6_concordance)도 함께 요청한다.

캐시: data/raw/comtrade/<reporter>_<year>.json
산출: research/outputs/comtrade_mirror_hs6_2012_2024.csv
      (reporter, reporterISO, year, cmdCode, classification, value_usd, net_kg, qty, qty_unit)
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import duckdb
import pandas as pd
import requests

import os
ROOT = Path(__file__).resolve().parents[1]                      # KCSTARIFF/research
KCSDB2 = Path(os.environ.get("KCSDB2_ROOT", r"C:\Work\Projects\KCSDB2"))
KCS = KCSDB2 / "data" / "processed" / "kcsdb.duckdb"
OUT = ROOT / "outputs"
RAW = ROOT / "data" / "raw" / "comtrade"
URL = "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
# 관세청 stat_cd → Comtrade reporterCode (UN M49). 대만은 490(Other Asia, nes).
REPORTERS = {"CN": 156, "US": 842, "JP": 392, "VN": 704, "AU": 36, "MY": 458, "DE": 276, "SG": 702, "BR": 76, "ID": 360,
             "AR": 32, "TH": 764, "NL": 528, "RU": 643, "TW": 490, "AE": 784, "FR": 250, "IT": 380, "NZ": 554, "IN": 699,
             "CA": 124, "GB": 826, "CH": 756, "ES": 724, "PE": 604, "MX": 484, "PH": 608, "TR": 792, "CL": 152, "EG": 818}
YEARS = range(2012, 2025)


def scope() -> tuple[list[str], pd.DataFrame]:
    pairs = pd.read_csv(OUT / "코드쌍_목록.csv", dtype={"hs10_high": str, "hs10_low": str})
    pairs = pairs[(pairs.확인.fillna("") == "Y") | pairs.source.isin(["별표", "별표(선택)"])]
    treat = pd.read_csv(OUT / "처리군_목록.csv", dtype={"hs10": str, "mate_hs10": str})
    tr = treat[treat.type.isin(["감시형", "감시형(혼합 전신)", "세율형"])]
    codes = set(pairs.hs10_high) | set(pairs.hs10_low) | set(tr.hs10) | set(tr.mate_hs10.dropna())
    hs6 = sorted({c[:6] for c in codes})
    con = duckdb.connect(); con.execute(f"ATTACH '{KCS.as_posix()}' AS s (READ_ONLY)")
    con.register("h6", pd.DataFrame({"hs6": hs6}))
    past = con.sql("SELECT DISTINCT hs2022, hs_past, past_version FROM s.dim_hs6_concordance JOIN h6 ON hs2022=hs6").df()
    con.close()
    return hs6, past


def codes_for_year(hs6: list[str], past: pd.DataFrame, year: int) -> list[str]:
    ver = "2012" if year <= 2016 else "2017" if year <= 2021 else None
    extra = set(past[past.past_version == ver].hs_past) if ver else set()
    return sorted(set(hs6) | extra)


def fetch(sess: requests.Session, reporter: int, year: int, codes: list[str], delay: float = 1.5) -> dict:
    """500행 상한에 걸리면 코드를 반으로 나눠 다시 받아 합친다. 429는 30초 쉬고 두 번 더 시도한다."""
    for attempt in range(3):
        r = sess.get(URL, params=dict(reporterCode=reporter, partnerCode=410, period=str(year), cmdCode=",".join(codes), flowCode="X"), timeout=120)
        if r.status_code == 429:
            time.sleep(30 * (attempt + 1)); continue
        r.raise_for_status(); j = r.json(); break
    else:
        r.raise_for_status()
    if j.get("count", 0) >= 500 and len(codes) > 1:
        h = len(codes) // 2; time.sleep(delay)
        a = fetch(sess, reporter, year, codes[:h], delay); time.sleep(delay); b = fetch(sess, reporter, year, codes[h:], delay)
        return {"count": a.get("count", 0) + b.get("count", 0), "data": a.get("data", []) + b.get("data", []), "split": True}
    return j


def main() -> None:
    ap = argparse.ArgumentParser(); ap.add_argument("--delay", type=float, default=1.5); ap.add_argument("--parse-only", action="store_true"); ap.add_argument("--refetch-capped", action="store_true", help="500행에 걸린 캐시를 나눠 다시 받는다")
    args = ap.parse_args()
    hs6, past = scope()
    RAW.mkdir(parents=True, exist_ok=True)
    sess = requests.Session(); sess.headers.update({"User-Agent": "Mozilla/5.0"})
    rows, failed = [], []
    for iso, rep in REPORTERS.items():
        for y in YEARS:
            f = RAW / f"{iso}_{y}.json"
            if f.exists() and not args.refetch_capped:
                j = json.loads(f.read_text(encoding="utf-8"))
            elif f.exists() and json.loads(f.read_text(encoding="utf-8")).get("count", 0) < 500:
                j = json.loads(f.read_text(encoding="utf-8"))
            elif args.parse_only:
                continue
            else:
                try:
                    j = fetch(sess, rep, y, codes_for_year(hs6, past, y), args.delay)
                except Exception as e:  # noqa: BLE001
                    failed.append((iso, y, str(e)[:80])); time.sleep(args.delay * 4); continue
                f.write_text(json.dumps(j, ensure_ascii=False), encoding="utf-8"); time.sleep(args.delay)
            for d in j.get("data", []):
                rows.append(dict(reporter=iso, reporterISO=d.get("reporterISO"), year=y, cmdCode=d["cmdCode"], classification=d.get("classificationCode"),
                                 value_usd=d.get("primaryValue"), net_kg=d.get("netWgt"), qty=d.get("qty"), qty_unit=d.get("qtyUnitAbbr"), count=j.get("count")))
        print(f"{iso}: 행 {sum(1 for r in rows if r['reporter']==iso):,}, 실패 {len(failed)}", flush=True)
    df = pd.DataFrame(rows)
    df.to_csv(OUT / "comtrade_mirror_hs6_2012_2024.csv", index=False, encoding="utf-8-sig")
    capped = int(sum(1 for iso in REPORTERS for y in YEARS if (RAW / f"{iso}_{y}.json").exists() and json.loads((RAW / f"{iso}_{y}.json").read_text(encoding="utf-8")).get("count", 0) >= 500 and not json.loads((RAW / f"{iso}_{y}.json").read_text(encoding="utf-8")).get("split")))
    print(f"완료: {len(df):,}행, 보고국 {df.reporter.nunique()}, HS6 {df.cmdCode.nunique()}, 500행 상한에 걸린 채 남은 호출 {capped}, 실패 {len(failed)}")
    if failed:
        print(failed[:10])


if __name__ == "__main__":
    main()


## 9. 한국은행 ECOS — 원/달러 환율 월평균

부록 A3.3. 2026-09-13 복원 스크립트. 옛 산출과 237개월 전부 같다.

In [ ]:
# ===== 파일: research/scripts/26_fetch_ecos_fx.py (그대로 옮김, 2,599바이트) =====
"""
26_fetch_ecos_fx.py — 한국은행 ECOS 731Y001(주요국 통화의 대원화 환율)에서 원/미국달러 매매기준율을 일별로 받아 월평균을 만든다(자료 논문 부록 A3.3).

2026-09-12에 대화형으로 만든 outputs/환율_월별_USDKRW.csv 를 2026-09-13에 스크립트로 복원했다. 복원본은 옛 CSV와 237개월 전부 같다(소수 2자리).
이 통계표는 월별 주기 조회가 없고 일별만 있으므로 해마다 일별 전부를 받아(한 해 1,000행 상한 안) 달마다 평균한다.
항목 코드는 StatisticItemList 로 확인한다(0000001 = 원/미국달러(매매기준율)). 키는 KCSDB2/config/api_key.env 의 ECOS_API_KEY (저장소에 올리지 않는다).
"""
from __future__ import annotations

import os
import time
from pathlib import Path

import pandas as pd
import requests

ROOT = Path(__file__).resolve().parents[1]
KCSDB2 = Path(os.environ.get("KCSDB2_ROOT", r"C:\Work\Projects\KCSDB2"))
OUT = ROOT / "outputs" / "환율_월별_USDKRW.csv"
STAT = "731Y001"


def api_key() -> str:
    for line in open(KCSDB2 / "config" / "api_key.env", encoding="utf-8"):
        if line.startswith("ECOS_API_KEY"):
            return line.split("=", 1)[1].strip().strip('"').strip("'")
    raise SystemExit("ECOS_API_KEY 없음")


def main(y0: int = 2007, y1: int = 2026) -> None:
    key = api_key()
    items = requests.get(f"https://ecos.bok.or.kr/api/StatisticItemList/{key}/json/kr/1/100/{STAT}", timeout=60).json()["StatisticItemList"]["row"]
    code = next(it["ITEM_CODE"] for it in items if "미국달러" in it["ITEM_NAME"])
    rows = []
    for y in range(y0, y1 + 1):
        for start in (1, 1001):
            r = requests.get(f"https://ecos.bok.or.kr/api/StatisticSearch/{key}/json/kr/{start}/{start + 999}/{STAT}/D/{y}0101/{y}1231/{code}", timeout=60).json()
            data = r.get("StatisticSearch", {}).get("row", [])
            rows += [(d["TIME"], float(d["DATA_VALUE"])) for d in data if d.get("DATA_VALUE") not in (None, "", "-")]
            if len(data) < 1000:
                break
            time.sleep(0.3)
        time.sleep(0.3)
    fx = pd.DataFrame(rows, columns=["date", "rate"]).drop_duplicates("date")
    fx["yyyymm"] = fx.date.str[:6].astype(int)
    mon = fx.groupby("yyyymm").rate.mean().round(2).reset_index().rename(columns={"rate": "krw_per_usd"})
    mon["year"] = mon.yyyymm // 100
    mon.to_csv(OUT, index=False, encoding="utf-8-sig")
    print(f"{OUT.name}: {len(mon)}개월 ({mon.yyyymm.min()}~{mon.yyyymm.max()}), 일별 {len(fx):,}행")


if __name__ == "__main__":
    main()


## 10. 법령 대조 표본 214개 — 관세율표 HWP와 조정관세 별표 PDF

본문 VI장 다섯째, 부록 A2.5.

In [ ]:
# ===== 파일: research/scripts/24_check_tariff_annex.py (그대로 옮김, 9,331바이트) =====
"""
24_check_tariff_annex.py — 법령 대조 표본 214개의 확인(자료 논문 VI장 다섯째 검증, 2026-09-13).

입력
  data/external/관세법_별표_관세율표/관세율표_개정2022-12-31_시행2025-01-01_lsiSeq17031935.hwp
      관세법 별표 관세율표, 2025-01-01 시행 판본(별표 개정 2022-12-31). 법령센터 별표 페이지의 판본별 주소
      http://www.law.go.kr/BYL/grtFile/law0015562022123119186KC_000000E.hwp 에서 받았다(HWP 5.0, 1.3MB).
  data/external/조정관세_규정_별표/조정관세_별표_개정2024-12-31_2025년적용_flSeq147597553.pdf
      관세법 제69조에 따른 조정관세의 적용에 관한 규정 [별표] <개정 2024.12.31>, 2025년 적용.
      https://www.law.go.kr/LSW/flDownload.do?gubun=&flSeq=147597553&bylClsCd=110201
  outputs/양허관세_별표1_2025.csv (별표 1의 가·나, 2.2절), outputs/실행세율_법령대조.csv (scripts/02가 만든 표본 214개)

하는 일
  1) 관세율표 HWP의 본문 텍스트를 읽어(olefile·zlib, PARA_TEXT) 소호(호 4자리 + 소호 2자리)마다 세율을 뽑는다. 소호 아래에
     "1. …", "가. …", "1) …" 세 단계의 세부 항목이 있으면 항목마다 (라벨 경로, 세율)을 잎으로 둔다 → 관세율표_2025_소호항목_세율.csv
  2) 표본 214개 전부의 기본세율(r_A)을 그 소호의 잎 세율과 맞댄다. 소호에 세율이 하나면 그 값과, 여럿이면 표본 코드 품명에 맞는 항목의
     값과 같은지 본다(항목 라벨을 비고에 남긴다).
  3) 실행세율 규정별로 법령 값을 맞댄다 — W2는 별표 1의 나(또는 가), C는 별표 1의 가, L은 조정관세 별표(2103.90의 45%).
  4) 실행세율_법령대조.csv의 빈 열(법령_확인세율·확인_출처·비고)을 채우고 실행세율_법령대조_요약.csv를 쓴다(노트북 §1b가 읽는다).
"""
from __future__ import annotations

import re
import struct
import zlib
from pathlib import Path

import fitz  # pymupdf
import numpy as np
import olefile
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
OUT = ROOT / "outputs"
HWP = ROOT / "data" / "external" / "관세법_별표_관세율표" / "관세율표_개정2022-12-31_시행2025-01-01_lsiSeq17031935.hwp"
PDF = ROOT / "data" / "external" / "조정관세_규정_별표" / "조정관세_별표_개정2024-12-31_2025년적용_flSeq147597553.pdf"
SRC_A = "관세법 별표 관세율표(개정 2022-12-31, 2025-01-01 시행 판본)"
SRC_L = "관세법 제69조에 따른 조정관세의 적용에 관한 규정 별표(개정 2024-12-31, 2025년 적용)"


def hwp_text(path: Path) -> list[str]:
    ole = olefile.OleFileIO(str(path))
    compressed = bool(ole.openstream("FileHeader").read()[36] & 1)
    out = []
    for entry in sorted(ole.listdir()):
        if entry[0] != "BodyText":
            continue
        data = ole.openstream(entry).read()
        if compressed:
            data = zlib.decompress(data, -15)
        i = 0
        while i + 4 <= len(data):
            head = struct.unpack("<I", data[i:i + 4])[0]
            tag, size = head & 0x3FF, (head >> 20) & 0xFFF
            i += 4
            if size == 0xFFF:
                size = struct.unpack("<I", data[i:i + 4])[0]
                i += 4
            if tag == 67:
                out.append(data[i:i + size].decode("utf-16le", errors="ignore"))
            i += size
    lines = [re.sub(r"[\x00-\x1f]", " ", l).strip() for l in out]
    return [l for l in lines if l]


IS_RATE = re.compile(r"(\d+(\.\d+)?|무세|자유)(\s*\(.*\))?$")
L1, L2, L3 = re.compile(r"^\d+\.\s"), re.compile(r"^[가-힣]\.\s"), re.compile(r"^\d+\)\s")


def parse_leaves(lines: list[str]) -> pd.DataFrame:
    """소호마다 (라벨 경로, 세율) 잎을 뽑는다. 표의 칸이 문단 순서로 나오므로 [호, 소호, 품명, 세율 | 항목·세율의 나열] 꼴이다."""
    is_rate = lambda s: IS_RATE.fullmatch(s) is not None
    is_item = lambda s: bool(L1.match(s) or L2.match(s) or L3.match(s))
    leaves, i, n = [], 0, len(lines)
    while i < n - 1:
        if re.fullmatch(r"\d{4}", lines[i]) and re.fullmatch(r"\d{2}", lines[i + 1]):
            hs6 = lines[i] + lines[i + 1]; j = i + 2; block = []
            while j < n and not (re.fullmatch(r"\d{4}", lines[j]) and j + 1 < n and (re.fullmatch(r"\d{2}", lines[j + 1]) or not is_rate(lines[j + 1]))):
                block.append(lines[j]); j += 1
            name = block[0] if block else ""; rest = block[1:]
            if not rest and is_rate(name):
                leaves.append((hs6, "", name))
            elif rest and not any(is_item(x) for x in rest):
                r = [x for x in rest if is_rate(x)]; leaves.append((hs6, name, r[0] if r else ""))
            else:
                l1 = l2 = ""; k = 0
                while k < len(rest):
                    x = rest[k]
                    if L1.match(x): l1, l2 = x, ""
                    elif L2.match(x): l2 = x
                    if is_item(x):
                        lab = x if L1.match(x) else (l1 + " > " + x if L2.match(x) else l1 + " > " + l2 + " > " + x)
                        if k + 1 < len(rest) and is_rate(rest[k + 1]):
                            leaves.append((hs6, lab, rest[k + 1])); k += 2; continue
                    elif is_rate(x) and k == 0:
                        leaves.append((hs6, name, x))
                    k += 1
            i = j
        else:
            i += 1
    lv = pd.DataFrame(leaves, columns=["hs6", "label", "rate_txt"])
    lv["rate"] = lv.rate_txt.map(rate_num)
    return lv


def rate_num(x) -> float:
    m = re.match(r"^(\d+(\.\d+)?)", str(x))
    return float(m.group(1)) if m else (0.0 if str(x).startswith(("무세", "자유")) else np.nan)


def main() -> None:
    lines = hwp_text(HWP)
    lv = parse_leaves(lines)
    lv.to_csv(HWP.parent / "관세율표_2025_소호항목_세율.csv", index=False, encoding="utf-8-sig")
    print(f"관세율표: 소호 {lv.hs6.nunique():,}개, 세율 항목 {len(lv):,}개, 세율 결측 {int(lv.rate.isna().sum())}")

    d = pd.read_csv(OUT / "실행세율_법령대조.csv", dtype=str, keep_default_na=False)
    by = pd.read_csv(OUT / "양허관세_별표1_2025.csv", dtype={"hs10": str})
    adj = "\n".join(p.get_text() for p in fitz.open(str(PDF)))
    adj_2103 = adj[adj.find("2103"):][:600]
    assert "45%" in adj_2103 and "고추" in adj_2103, "조정관세 별표에서 2103.90 45%를 찾지 못했다"

    status_a = {}
    for r in d.itertuples():
        hs6 = r.hs10[:6]; rA = float(r.r_A); g = lv[lv.hs6 == hs6]; rates = sorted(set(g.rate.dropna()))
        hit = g[np.isclose(g.rate, rA, atol=0.01)]
        st = "일치(단일)" if len(rates) == 1 and np.isclose(rates[0], rA, atol=0.01) else ("항목 일치" if len(hit) else "불일치")
        status_a[r.hs10] = (st, " | ".join(hit.label.tolist())[:60])
    n_single = sum(1 for s, _ in status_a.values() if s == "일치(단일)"); n_item = sum(1 for s, _ in status_a.values() if s == "항목 일치")
    n_bad = sum(1 for s, _ in status_a.values() if s == "불일치")

    for i, r in d.iterrows():
        st, lab = status_a[r.hs10]; note_a = f"기본세율 {float(r.r_A):g} {st}" + (f"({lab})" if st == "항목 일치" else "")
        if r.mfn_regime == "A":
            d.at[i, "법령_확인세율"] = f"{float(r.r_A):g}"; d.at[i, "확인_출처"] = SRC_A; d.at[i, "비고"] = note_a + "(2026-09-13)"
        elif r.mfn_regime in ("W2", "C"):
            col = "r_W2" if r.mfn_regime == "W2" else "r_C"; rows = by[by.hs10 == r.hs10]
            hit = rows[np.isclose(rows.adval.astype(float), float(r[col]), atol=0.01)]
            assert len(hit), r.hs10
            d.at[i, "법령_확인세율"] = f"{float(r[col]):g}"; d.at[i, "확인_출처"] = f"양허관세 규정 별표 {hit.byeolpyo.iloc[0]}(2024-12-31 개정, 2025년 적용); " + SRC_A
            d.at[i, "비고"] = ("양허 세율 일치" if r.mfn_regime == "W2" else "WTO 협정세율 일치") + "(2026-09-13); " + note_a
        elif r.mfn_regime == "L":
            assert r.hs10.startswith("210390") and float(r.r_L) == 45.0, r.hs10
            d.at[i, "법령_확인세율"] = "45"; d.at[i, "확인_출처"] = SRC_L + "; " + SRC_A
            d.at[i, "비고"] = "조정관세 45%(2103.90 중 고추·마늘·양파·생강 함량 각 20% 이상 또는 합 40% 이상인 것) 일치; " + note_a + "(2026-09-13)"
        else:
            raise ValueError(r.mfn_regime)
    d.to_csv(OUT / "실행세율_법령대조.csv", index=False, encoding="utf-8-sig")
    vc = d.mfn_regime.value_counts()
    summ = pd.DataFrame([("표본", len(d)), ("확인", int((d.법령_확인세율 != "").sum())), ("양허 W2 일치", int(vc.get("W2", 0))), ("WTO C 일치", int(vc.get("C", 0))), ("기본세율 A 일치", int(vc.get("A", 0))), ("조정관세 L 일치", int(vc.get("L", 0))),
                         ("기본세율 단일 소호 일치", n_single), ("기본세율 세부 항목 일치", n_item), ("어긋남", n_bad), ("관세율표 소호 수", lv.hs6.nunique()), ("관세율표 세율 항목 수", len(lv))], columns=["item", "value"])
    summ.to_csv(OUT / "실행세율_법령대조_요약.csv", index=False, encoding="utf-8-sig")
    print(summ.to_string(index=False))


if __name__ == "__main__":
    main()
